# Stage 2: Reward Model (Bradley-Terry) on GPT-2

In [ ]:
import os
import torch
import torch.nn as nn
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
)
from transformers.modeling_outputs import SequenceClassifierOutput
from sklearn.metrics import accuracy_score
import numpy as np
import random

# ---------- Paths ----------
SFT_CHECKPOINT = "./sft_gpt2_dolly/final"
RM_OUTPUT_DIR  = "./rm_gpt2_hh"
os.makedirs(RM_OUTPUT_DIR, exist_ok=True)

MAX_SEQ_LEN = 256
NUM_EPOCHS = 2
PER_DEVICE_BATCH = 4
GRAD_ACCUM = 8
LR = 1e-5
SEED = 42
MAX_PAIRS = 12000

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# ---------- Tokenizer ----------
tokenizer = AutoTokenizer.from_pretrained(SFT_CHECKPOINT, local_files_only=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# ---------- Data ----------
print("Loading HH-RLHF...")
raw = load_dataset("Anthropic/hh-rlhf", data_dir="helpful-base", split="train")
pairs = raw.shuffle(seed=SEED).select(range(min(MAX_PAIRS, len(raw))))
pairs = pairs.train_test_split(test_size=0.1, seed=SEED)
train_pairs, val_pairs = pairs["train"], pairs["test"]

def tokenize(examples):
    c = tokenizer(examples["chosen"], truncation=True, max_length=MAX_SEQ_LEN, padding=False)
    r = tokenizer(examples["rejected"], truncation=True, max_length=MAX_SEQ_LEN, padding=False)
    return {
        "input_ids_chosen": c["input_ids"],
        "attention_mask_chosen": c["attention_mask"],
        "input_ids_rejected": r["input_ids"],
        "attention_mask_rejected": r["attention_mask"],
    }

train_pairs = train_pairs.map(tokenize, batched=True, remove_columns=train_pairs.column_names)
val_pairs   = val_pairs.map(tokenize, batched=True, remove_columns=val_pairs.column_names)

# ---------- Model ----------
model = AutoModelForSequenceClassification.from_pretrained(
    SFT_CHECKPOINT,
    num_labels=1,
    problem_type="regression",
    local_files_only=True,
    ignore_mismatched_sizes=True,
)
model.config.pad_token_id = tokenizer.pad_token_id
model.config.model_type = "gpt2"               # force it
model.gradient_checkpointing_enable()

# ---------- Collator ----------
class PairCollator:
    def __init__(self, tok):
        self.tok = tok
    def __call__(self, features):
        batch_c = self.tok.pad(
            {"input_ids": [f["input_ids_chosen"] for f in features],
             "attention_mask": [f["attention_mask_chosen"] for f in features]},
            return_tensors="pt", padding=True)
        batch_r = self.tok.pad(
            {"input_ids": [f["input_ids_rejected"] for f in features],
             "attention_mask": [f["attention_mask_rejected"] for f in features]},
            return_tensors="pt", padding=True)
        return {
            "input_ids_chosen": batch_c["input_ids"],
            "attention_mask_chosen": batch_c["attention_mask"],
            "input_ids_rejected": batch_r["input_ids"],
            "attention_mask_rejected": batch_r["attention_mask"],
        }

# ---------- Trainer with BT loss ----------
class RewardTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        r_c = model(input_ids=inputs["input_ids_chosen"],
                    attention_mask=inputs["attention_mask_chosen"]).logits.squeeze(-1)
        r_r = model(input_ids=inputs["input_ids_rejected"],
                    attention_mask=inputs["attention_mask_rejected"]).logits.squeeze(-1)
        loss = -torch.nn.functional.logsigmoid(r_c - r_r).mean()
        if return_outputs:
            return loss, {"rewards_chosen": r_c, "rewards_rejected": r_r}
        return loss

    def prediction_step(self, model, inputs, prediction_loss_only, ignore_keys=None):
        with torch.no_grad():
            loss, outputs = self.compute_loss(model, inputs, return_outputs=True)
            preds = (outputs["rewards_chosen"] > outputs["rewards_rejected"]).float()
            labels = torch.ones_like(preds)
        return (loss, preds, labels)

def compute_metrics(eval_pred):
    preds, labels = eval_pred
    return {"accuracy": accuracy_score(labels, preds)}

# ---------- Train ----------
args = TrainingArguments(
    output_dir=RM_OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=PER_DEVICE_BATCH,
    per_device_eval_batch_size=PER_DEVICE_BATCH,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LR,
    fp16=True,
    logging_steps=50,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model="eval_accuracy",
    greater_is_better=True,
    report_to="none",
    seed=SEED,
    remove_unused_columns=False,
)

trainer = RewardTrainer(
    model=model,
    args=args,
    train_dataset=train_pairs,
    eval_dataset=val_pairs,
    data_collator=PairCollator(tokenizer),
    compute_metrics=compute_metrics,
)

print("Starting Reward Model training...")
trainer.train()

# ---------- FORCE correct save ----------
final_path = os.path.join(RM_OUTPUT_DIR, "final")
os.makedirs(final_path, exist_ok=True)

# Save model + config properly
model.save_pretrained(final_path)
tokenizer.save_pretrained(final_path)

# Extra safety: make sure model_type is written
import json
cfg_path = os.path.join(final_path, "config.json")
with open(cfg_path) as f:
    cfg = json.load(f)
cfg["model_type"] = "gpt2"
cfg["architectures"] = ["GPT2ForSequenceClassification"]
with open(cfg_path, "w") as f:
    json.dump(cfg, f, indent=2)

print(f"\nReward Model properly saved to: {final_path}")
print("Files now present:", os.listdir(final_path))